In [1]:
# !pip install "hopsworks[python]"

In [2]:
# !pip install python-dotenv

In [3]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

os.makedirs("/tmp", exist_ok=True)

In [4]:
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "Feature_Pipeline" else CWD
RAW_DIR = PROJECT_ROOT / "Data" / "Raw"
PROCESSED_DIR = PROJECT_ROOT / "Data" / "Processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_ROOT / ".env")

key = os.getenv("HOPSWORK_KEY")

In [5]:
import hopsworks

project = hopsworks.login(
    project='aqi_karachi_samu_2026',
    host="eu-west.cloud.hopsworks.ai",
    port=443,
    api_key_value=key,
)

fs = project.get_feature_store()
print(f"Connected to {project.name}")

2026-08-22 21:12:57,319 INFO: Initializing external client
2026-08-22 21:12:57,320 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-08-22 21:13:02,020 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/41193
Connected to aqi_karachi_samu_2026


# DAY ONE AQI

In [6]:
df = pd.read_csv(PROCESSED_DIR / "Day_One_AQI.csv")

In [7]:
df.head()

,time,temperature_2m,wind_speed_10m,relative_humidity_2m,surface_pressure,boundary_layer_height,dew_point_2m,precipitation,cloud_cover,wind_direction_10m,...,f24_boundary_layer_height,f24_dew_point_2m,f24_precipitation,f24_cloud_cover,f24_wind_gusts_10m,f24_shortwave_radiation,wind_pollution_dispersion,pressure_change_3hour,Humidity_Level,Day_1_Future_AQI
0,2023-08-04 00:00:00+05:00,28.441667,20.516667,83.458333,1000.087500,752.708333,25.291667,0.037500,50.083333,245.083333,...,818.333333,24.866667,0.037500,69.833333,33.450000,220.291667,15605.187500,0.225000,3.150000,79.458333
1,2023-08-05 00:00:00+05:00,28.329167,21.941667,81.750000,1002.600000,818.333333,24.866667,0.037500,69.833333,247.750000,...,900.416667,24.479167,0.029167,81.625000,45.562500,175.958333,18252.291667,0.266667,3.462500,78.833333
2,2023-08-06 00:00:00+05:00,28.233333,30.679167,80.375000,1001.608333,900.416667,24.479167,0.029167,81.625000,254.500000,...,916.458333,24.266667,0.070833,75.000000,49.770833,171.375000,27933.375000,-0.387500,3.754167,78.875000
3,2023-08-07 00:00:00+05:00,28.041667,33.220833,80.208333,998.991667,916.458333,24.266667,0.070833,75.000000,250.250000,...,805.416667,24.512500,0.091667,68.541667,44.275000,172.000000,30741.333333,-0.200000,3.775000,78.791667
4,2023-08-08 00:00:00+05:00,27.995833,29.429167,81.666667,999.362500,805.416667,24.512500,0.091667,68.541667,245.916667,...,798.750000,24.487500,0.033333,72.916667,43.120833,220.875000,23936.687500,0.304167,3.483333,73.416667


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1092 entries, 0 to 1091
Data columns (total 50 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   time                       1092 non-null   object 
 1   temperature_2m             1092 non-null   float64
 2   wind_speed_10m             1092 non-null   float64
 3   relative_humidity_2m       1092 non-null   float64
 4   surface_pressure           1092 non-null   float64
 5   boundary_layer_height      911 non-null    float64
 6   dew_point_2m               1092 non-null   float64
 7   precipitation              1092 non-null   float64
 8   cloud_cover                1092 non-null   float64
 9   wind_direction_10m         1092 non-null   float64
 10  wind_gusts_10m             1092 non-null   float64
 11  shortwave_radiation        1092 non-null   float64
 12  pm10                       1092 non-null   float64
 13  pm2_5                      1092 non-null   float

In [9]:
df['time'] = pd.to_datetime(df['time']).dt.tz_localize(None)

In [10]:
df.columns = [c.lower() for c in df.columns]

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1092 entries, 0 to 1091
Data columns (total 50 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   time                       1092 non-null   datetime64[ns]
 1   temperature_2m             1092 non-null   float64       
 2   wind_speed_10m             1092 non-null   float64       
 3   relative_humidity_2m       1092 non-null   float64       
 4   surface_pressure           1092 non-null   float64       
 5   boundary_layer_height      911 non-null    float64       
 6   dew_point_2m               1092 non-null   float64       
 7   precipitation              1092 non-null   float64       
 8   cloud_cover                1092 non-null   float64       
 9   wind_direction_10m         1092 non-null   float64       
 10  wind_gusts_10m             1092 non-null   float64       
 11  shortwave_radiation        1092 non-null   float64       
 12  pm10  

# Day 2 AQI

In [12]:
df1 = pd.read_csv(PROCESSED_DIR / "Cyclic_Day_Two_AQI.csv")

In [13]:
df1.head()

,time,temperature_2m,wind_speed_10m,relative_humidity_2m,surface_pressure,boundary_layer_height,dew_point_2m,precipitation,cloud_cover,wind_direction_10m,...,Day_2_Future_AQI,cyclic_day_sin,cyclic_day_cos,cyclic_week_sin,cyclic_week_cos,delta_temperature_2m,delta_wind_speed_10m,delta_surface_pressure,delta_relative_humidity_2m,delta_boundary_layer_height
0,2023-08-04 00:00:00+05:00,28.441667,20.516667,83.458333,1000.087500,752.708333,25.291667,0.037500,50.083333,245.083333,...,78.833333,-0.545240,-0.838280,-0.433884,-0.900969,-0.208333,10.162500,1.520833,-3.083333,147.708333
1,2023-08-05 00:00:00+05:00,28.329167,21.941667,81.750000,1002.600000,818.333333,24.866667,0.037500,69.833333,247.750000,...,78.875000,-0.559589,-0.828770,-0.974928,-0.222521,-0.287500,11.279167,-3.608333,-1.541667,98.125000
2,2023-08-06 00:00:00+05:00,28.233333,30.679167,80.375000,1001.608333,900.416667,24.479167,0.029167,81.625000,254.500000,...,78.791667,-0.573772,-0.819015,-0.781831,0.623490,-0.237500,-1.250000,-2.245833,1.291667,-95.000000
3,2023-08-07 00:00:00+05:00,28.041667,33.220833,80.208333,998.991667,916.458333,24.266667,0.070833,75.000000,250.250000,...,73.416667,-0.587785,-0.809017,0.000000,1.000000,-0.070833,-4.783333,3.595833,1.416667,-117.708333
4,2023-08-08 00:00:00+05:00,27.995833,29.429167,81.666667,999.362500,805.416667,24.512500,0.091667,68.541667,245.916667,...,68.791667,-0.601624,-0.798779,0.781831,0.623490,-0.166667,-1.858333,4.245833,0.791667,-29.375000


In [14]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1090 entries, 0 to 1089
Data columns (total 53 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   time                         1090 non-null   object 
 1   temperature_2m               1090 non-null   float64
 2   wind_speed_10m               1090 non-null   float64
 3   relative_humidity_2m         1090 non-null   float64
 4   surface_pressure             1090 non-null   float64
 5   boundary_layer_height        909 non-null    float64
 6   dew_point_2m                 1090 non-null   float64
 7   precipitation                1090 non-null   float64
 8   cloud_cover                  1090 non-null   float64
 9   wind_direction_10m           1090 non-null   float64
 10  wind_gusts_10m               1090 non-null   float64
 11  shortwave_radiation          1090 non-null   float64
 12  pm10                         1090 non-null   float64
 13  pm2_5             

In [15]:
df1['time'] = pd.to_datetime(df1['time']).dt.tz_localize(None)

In [16]:
df1.columns = [c.lower() for c in df1.columns]

In [17]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1090 entries, 0 to 1089
Data columns (total 53 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   time                         1090 non-null   datetime64[ns]
 1   temperature_2m               1090 non-null   float64       
 2   wind_speed_10m               1090 non-null   float64       
 3   relative_humidity_2m         1090 non-null   float64       
 4   surface_pressure             1090 non-null   float64       
 5   boundary_layer_height        909 non-null    float64       
 6   dew_point_2m                 1090 non-null   float64       
 7   precipitation                1090 non-null   float64       
 8   cloud_cover                  1090 non-null   float64       
 9   wind_direction_10m           1090 non-null   float64       
 10  wind_gusts_10m               1090 non-null   float64       
 11  shortwave_radiation          1090 non-null 

# Day 3 AQI

In [18]:
df2 = pd.read_csv(PROCESSED_DIR / "Cyclic_Day_Three_AQI.csv")

In [19]:
df2.head()

,time,temperature_2m,wind_speed_10m,relative_humidity_2m,surface_pressure,boundary_layer_height,dew_point_2m,precipitation,cloud_cover,wind_direction_10m,...,Day_3_Future_AQI,cyclic_day_sin,cyclic_day_cos,cyclic_week_sin,cyclic_week_cos,delta_temperature_2m,delta_wind_speed_10m,delta_surface_pressure,delta_relative_humidity_2m,delta_boundary_layer_height
0,2023-08-04 00:00:00+05:00,28.441667,20.516667,83.458333,1000.087500,752.708333,25.291667,0.037500,50.083333,245.083333,...,78.875000,-0.545240,-0.838280,-0.433884,-0.900969,-0.400000,12.704167,-1.095833,-3.250000,163.750000
1,2023-08-05 00:00:00+05:00,28.329167,21.941667,81.750000,1002.600000,818.333333,24.866667,0.037500,69.833333,247.750000,...,78.791667,-0.559589,-0.828770,-0.974928,-0.222521,-0.333333,7.487500,-3.237500,-0.083333,-12.916667
2,2023-08-06 00:00:00+05:00,28.233333,30.679167,80.375000,1001.608333,900.416667,24.479167,0.029167,81.625000,254.500000,...,73.416667,-0.573772,-0.819015,-0.781831,0.623490,-0.262500,-2.241667,0.979167,1.250000,-101.666667
3,2023-08-07 00:00:00+05:00,28.041667,33.220833,80.208333,998.991667,916.458333,24.266667,0.070833,75.000000,250.250000,...,68.791667,-0.587785,-0.809017,0.000000,1.000000,-0.212500,-5.650000,4.616667,2.250000,-140.416667
4,2023-08-08 00:00:00+05:00,27.995833,29.429167,81.666667,999.362500,805.416667,24.512500,0.091667,68.541667,245.916667,...,69.916667,-0.601624,-0.798779,0.781831,0.623490,-0.275000,-3.066667,4.620833,0.625000,-81.250000


In [20]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1088 entries, 0 to 1087
Data columns (total 53 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   time                         1088 non-null   object 
 1   temperature_2m               1088 non-null   float64
 2   wind_speed_10m               1088 non-null   float64
 3   relative_humidity_2m         1088 non-null   float64
 4   surface_pressure             1088 non-null   float64
 5   boundary_layer_height        907 non-null    float64
 6   dew_point_2m                 1088 non-null   float64
 7   precipitation                1088 non-null   float64
 8   cloud_cover                  1088 non-null   float64
 9   wind_direction_10m           1088 non-null   float64
 10  wind_gusts_10m               1088 non-null   float64
 11  shortwave_radiation          1088 non-null   float64
 12  pm10                         1088 non-null   float64
 13  pm2_5             

In [21]:
df2['time'] = pd.to_datetime(df2['time']).dt.tz_localize(None)

In [22]:
df2.columns = [c.lower() for c in df2.columns]

In [23]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1088 entries, 0 to 1087
Data columns (total 53 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   time                         1088 non-null   datetime64[ns]
 1   temperature_2m               1088 non-null   float64       
 2   wind_speed_10m               1088 non-null   float64       
 3   relative_humidity_2m         1088 non-null   float64       
 4   surface_pressure             1088 non-null   float64       
 5   boundary_layer_height        907 non-null    float64       
 6   dew_point_2m                 1088 non-null   float64       
 7   precipitation                1088 non-null   float64       
 8   cloud_cover                  1088 non-null   float64       
 9   wind_direction_10m           1088 non-null   float64       
 10  wind_gusts_10m               1088 non-null   float64       
 11  shortwave_radiation          1088 non-null 

# Feature Group

In [24]:
DATASET_INSERT = {
    "aqi_daily_day1": (df,  "day_1_future_aqi"),
    "aqi_daily_day2": (df1, "day_2_future_aqi"),
    "aqi_daily_day3": (df2, "day_3_future_aqi"),
}

In [25]:
feature_group_object = {}
IN_Insert = [ ]

for feature_name, (data, target) in DATASET_INSERT.items():
    fg = fs.get_or_create_feature_group(
    name = feature_name,
    version = 1,
    description = f"Karachi AQI Daily Prediction of target {target}",
    primary_key = ["time"],
    event_time = "time",
    online_enabled = False,
    time_travel_format = "HUDI",
    )
    feature_group_object[feature_name] = fg
    if feature_name in IN_Insert:  
       fg.insert(data, wait=True)

# Verify

In [28]:
# feature_group_object["aqi_daily_day2"].read().shape

### Day 2 done do it for Day 3

In [29]:
# fg3 = fs.get_or_create_feature_group(
#     name = "aqi_daily_day3",
#     version = 1,
#     description = "Karachi AQI Daily Prediction of target aqi_daily_day3",
#     primary_key = ["time"],
#     event_time = "time",
#     online_enabled = False,
#     time_travel_format = "HUDI",
#     )
# feature_group_object["aqi_daily_day3"] = fg3
# fg3.insert(df2)

In [30]:
# fg3.read().shape

# Feature View

In [31]:
feature_view_object = {}

for feature_name, (data, target) in DATASET_INSERT.items():
    fg = feature_group_object[feature_name]
    fv = fs.get_or_create_feature_view(
    name = feature_name.replace("aqi_daily_","view_aqi_"),
    version = 1,
    description = f"Feature View of Karachi AQI {target}",
    query = fg.select_all(),
    labels = [target]
    )
    feature_view_object[feature_name] = fv
    print(fv.name, "done")

Feature view created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/41193/fs/30899/fv/view_aqi_day1/version/1
view_aqi_day1 done
Feature view created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/41193/fs/30899/fv/view_aqi_day2/version/1
view_aqi_day2 done
Feature view created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/41193/fs/30899/fv/view_aqi_day3/version/1
view_aqi_day3 done
